# Amphion — Phase 4: Train the Generator (VAE)

**This notebook runs on a free GPU (Kaggle) and is self-contained** — it fetches its
own data and needs nothing from the Amphion repo.

### What it does
Trains a small character-level **sequence VAE** on AMP positives (GRAMPA), then
**samples novel candidate peptides**, enforces a validity gate, and saves two files.

### ▶ How to run (Kaggle)
1. **Settings (right panel):** Accelerator = **GPU** (T4/P100), Internet = **ON**.
2. **Run All.** Training is a few minutes on GPU.
3. **Download both outputs** from the right panel (`/kaggle/working/`):
   - `generator_vae.pt`  → put in the repo's `models/`
   - `candidates.csv`    → put in the repo's `data/interim/`
4. Tell Claude Code to continue with Phase 5.

> Outputs are computational designs, not validated molecules.


In [ ]:
import os, time, urllib.request
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected — turn on the GPU accelerator (Settings) for a fast run.')

# Hyperparameters (tunable)
MAX_RES   = 50          # max residues (AMPs are short)
EMBED     = 64
HIDDEN    = 256
LATENT    = 32
BATCH     = 128
EPOCHS    = 60
LR        = 1e-3
KL_ANNEAL = 25          # epochs to ramp KL weight
BETA_MAX  = 0.5         # max KL weight (lower => stronger reconstruction/diversity)
N_CANDIDATES = 5000     # valid novel candidates to generate

## 1. Get AMP training data (GRAMPA positives)

In [ ]:
URL = 'https://raw.githubusercontent.com/zswitten/Antimicrobial-Peptides/master/data/grampa.csv'
urllib.request.urlretrieve(URL, 'grampa.csv')
g = pd.read_csv('grampa.csv')
g['sequence'] = g['sequence'].astype(str).str.upper().str.strip()
CANON = set('ACDEFGHIKLMNPQRSTVWY')
positives = [s for s in dict.fromkeys(g['sequence']) if s and all(c in CANON for c in s) and 5 <= len(s) <= MAX_RES]
print('unique AMP training sequences:', len(positives))
print('example:', positives[0])

## 2. Tokenize
Vocabulary = 3 special tokens (PAD/SOS/EOS) + 20 amino acids.

In [ ]:
PAD, SOS, EOS = 0, 1, 2
ITOS = ['<pad>', '<sos>', '<eos>'] + list('ACDEFGHIKLMNPQRSTVWY')
STOI = {c: i for i, c in enumerate(ITOS)}
VOCAB = len(ITOS)  # 23

def encode_seq(seq, max_res=MAX_RES):
    toks = [SOS] + [STOI[c] for c in seq] + [EOS]
    toks = toks[:max_res + 2]
    toks += [PAD] * (max_res + 2 - len(toks))
    return toks

def decode_tokens(tokens):
    out = []
    for t in tokens:
        t = int(t)
        if t == EOS: break
        if t >= 3: out.append(ITOS[t])
    return ''.join(out)

X = torch.tensor([encode_seq(s) for s in positives], dtype=torch.long)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X), batch_size=BATCH, shuffle=True)
print('token tensor:', tuple(X.shape))

## 3. Define the VAE
GRU encoder → latent z → GRU decoder (autoregressive). Identical to `src/amphion/generator/vae.py`.

In [ ]:
class PeptideVAE(nn.Module):
    def __init__(self, vocab=VOCAB, embed=EMBED, hidden=HIDDEN, latent=LATENT, max_residues=MAX_RES):
        super().__init__()
        self.hparams = dict(vocab=vocab, embed=embed, hidden=hidden, latent=latent, max_residues=max_residues)
        self.max_residues = max_residues
        self.emb = nn.Embedding(vocab, embed, padding_idx=PAD)
        self.enc_gru = nn.GRU(embed, hidden, batch_first=True)
        self.fc_mu = nn.Linear(hidden, latent)
        self.fc_logvar = nn.Linear(hidden, latent)
        self.lat2hid = nn.Linear(latent, hidden)
        self.dec_gru = nn.GRU(embed, hidden, batch_first=True)
        self.out = nn.Linear(hidden, vocab)
    def encode(self, x):
        _, h = self.enc_gru(self.emb(x)); h = h.squeeze(0)
        return self.fc_mu(h), self.fc_logvar(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def decode(self, z, x_in):
        h0 = torch.tanh(self.lat2hid(z)).unsqueeze(0)
        out, _ = self.dec_gru(self.emb(x_in), h0); return self.out(out)
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar)
        return self.decode(z, x[:, :-1]), mu, logvar
    @torch.no_grad()
    def sample(self, n, device, temperature=1.0, max_residues=None):
        self.eval(); max_residues = max_residues or self.max_residues
        z = torch.randn(n, self.hparams['latent'], device=device)
        h = torch.tanh(self.lat2hid(z)).unsqueeze(0)
        tok = torch.full((n, 1), SOS, dtype=torch.long, device=device)
        done = torch.zeros(n, dtype=torch.bool, device=device); seqs = [[] for _ in range(n)]
        for _ in range(max_residues + 1):
            out, h = self.dec_gru(self.emb(tok), h)
            logits = self.out(out[:, -1]) / max(temperature, 1e-6)
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1).squeeze(1)
            for i in range(n):
                if not done[i]:
                    v = int(nxt[i].item())
                    if v == EOS: done[i] = True
                    elif v >= 3: seqs[i].append(ITOS[v])
            tok = nxt.unsqueeze(1)
            if done.all(): break
        return [''.join(s) for s in seqs]

def vae_loss(logits, target, mu, logvar, beta):
    ce = F.cross_entropy(logits.reshape(-1, logits.size(-1)), target.reshape(-1), ignore_index=PAD)
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return ce + beta * kl, ce, kl

## 4. Train (with KL annealing)

In [ ]:
model = PeptideVAE().to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)
print('params:', sum(p.numel() for p in model.parameters()))
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train(); beta = BETA_MAX * min(1.0, epoch / KL_ANNEAL)
    tot = ce_t = kl_t = 0.0; nb = 0
    for (xb,) in loader:
        xb = xb.to(device)
        logits, mu, logvar = model(xb)
        loss, ce, kl = vae_loss(logits, xb[:, 1:], mu, logvar, beta)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        tot += loss.item(); ce_t += ce.item(); kl_t += kl.item(); nb += 1
    if epoch == 1 or epoch % 5 == 0:
        print(f'epoch {epoch:3d}  beta {beta:.2f}  loss {tot/nb:.3f}  ce {ce_t/nb:.3f}  kl {kl_t/nb:.3f}')
print(f'trained in {time.time()-t0:.0f}s')

## 5. Sample novel candidates + validity gate
Keep only canonical (guaranteed), length 5–50, **not** in training, and unique.

In [ ]:
known = set(positives); lo, hi = 5, MAX_RES
seen, kept = set(), []
while len(kept) < N_CANDIDATES:
    for s in model.sample(8000, device, temperature=1.0):
        if lo <= len(s) <= hi and s not in known and s not in seen:
            seen.add(s); kept.append(s)
            if len(kept) >= N_CANDIDATES: break
    print('collected', len(kept), '/', N_CANDIDATES)

def frac_kr(s): return sum(c in 'KR' for c in s) / len(s)
print('\nGenerated:', len(kept), '| mean length', round(np.mean([len(s) for s in kept]), 1))
print('mean K+R fraction  generated vs training:',
      round(np.mean([frac_kr(s) for s in kept]), 3), 'vs', round(np.mean([frac_kr(s) for s in positives]), 3),
      '  (cationic = AMP-like ✓)')
print('examples:')
for s in kept[:10]: print('  ', s)

## 6. Save artifacts → download both

In [ ]:
OUT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
torch.save({'hparams': model.hparams, 'state_dict': model.state_dict()}, f'{OUT}/generator_vae.pt')
pd.DataFrame({'sequence': kept, 'length': [len(s) for s in kept], 'source': 'vae'}).to_csv(f'{OUT}/candidates.csv', index=False)
print('SAVED to', OUT)
print('  generator_vae.pt  -> put in repo  models/')
print('  candidates.csv    -> put in repo  data/interim/')
print('Then tell Claude Code to continue with Phase 5.')